# Import Functions

In [3]:
!nvidia-smi

Tue Mar 11 16:57:56 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.144.03             Driver Version: 550.144.03     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               On  |   00000000:01:00.0 Off |                  Off |
| 30%   30C    P8             18W /  300W |    4057MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import sys 
!{sys.executable} -m pip install torchvision==0.17.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 128.7 MB/s eta 0:00:00
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.16.0+cu121
    Uninstalling torchvision-0.16.0+cu121:
      Successfully uninstalled torchvision-0.16.0+cu121


In [2]:
import sys 
!{sys.executable} -m pip install auto4dstem==0.2.7

In [4]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="1"
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import torch
from torch.autograd import Variable
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
import numpy as np 
import matplotlib.pyplot as plt 
from torch.autograd import Function
from collections import OrderedDict
import torch.nn as nn
import math
import pickle
import torch.autograd as autograd
import matplotlib.gridspec as gridspec
import os
import h5py
from tqdm import tqdm
from scipy import ndimage
import matplotlib.pyplot as plt
import skimage
from skimage.feature import peak_local_max
from skimage import data, img_as_float,feature
from skimage import io
import cv2

In [5]:
import torchvision
from torchvision import models, transforms, datasets

In [6]:
import matplotlib.pylab as pylab

params = {'axes.titlesize':20,
          'xtick.direction': 'in' ,
          'ytick.direction' : 'in',
          'xtick.top' : True,
          'ytick.right' : True,
          'ytick.labelsize':16,
          'xtick.labelsize':16
         }

pylab.rcParams.update(params)

In [7]:
torch.__version__

'2.2.0+cu121'

In [8]:
torchvision.__version__

'0.17.0+cu121'

In [9]:
#import atomai as aoi
#import kornia as K
import cv2
import scipy
import argparse
import skimage
from skimage.util import random_noise
from skimage import feature
import glob
from scipy import ndimage
import scipy as sp
import random

In [10]:
import warnings
warnings.filterwarnings('ignore') 

In [11]:
from functools import partial
import numpy as np
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import random_split

In [12]:
%load_ext autoreload
%autoreload 2
from auto4dstem.nn.Train_Function import TrainClass
from auto4dstem.Viz.util import mask_class
from auto4dstem.Viz.viz import set_format_Auto4D, visualize_simulate_result, visual_performance_plot,normalized_strain_matrices
# #from auto4dstem.Data.DataProcess import *
# from auto4dstem.nn.CC_ST_AE import *
# from auto4dstem.nn.Loss_Function import *
# from auto4dstem.nn.Train_Function import *
# from auto4dstem.Viz.util import *
# from auto4dstem.Viz.viz import *
from m3util.util.IO import make_folder
import warnings
warnings.filterwarnings('ignore') 

In [13]:
torch.cuda.device_count()

1

# Build two type of mask for two training process

In [14]:
def load_data(data_dir, w_bg=0.60):
    
    '''
    
        data_dir: path of the dataset
        label_index: path of the pretrained rotation 
    
    '''
    
    f = h5py.File(data_dir,'r')
    op4d = f['output4D']
    op4d = op4d[:,:,28:228,28:228]
    op4d = np.transpose(op4d, (1, 0, 3, 2))
    op4d = op4d.reshape(-1,200,200)
    f.close()
    
    if w_bg == 0:
        
        noisy_data = op4d*1e5/4
    
    else:
    
        noisy_data = np.zeros([65536,200,200])
        im=np.zeros([200,200])
        counts_per_probe = 1e5
        for i in tqdm(range(65536),leave=True,total=65536):
            test_img = np.copy(op4d[i])
            qx = np.fft.fftfreq( im.shape[0], d = 1)
            qy = np.fft.fftfreq( im.shape[1], d = 1)
            qya, qxa = np.meshgrid(qy, qx)
            qxa = np.fft.fftshift(qxa)
            qya = np.fft.fftshift(qya) 
            qra2 = qxa**2 + qya**2
            im_bg = 1./( 1 + qra2 / 1e-2**2 )
            im_bg = im_bg / np.sum(im_bg) 
            int_comb = test_img * (1 - w_bg) + im_bg * w_bg 
            int_noisy = np.random.poisson(int_comb * counts_per_probe) / counts_per_probe
            int_noisy = int_noisy*1e5/4
            noisy_data[i] = int_noisy
        
    del op4d
    
    noisy_data = noisy_data.reshape(-1,1,200,200)
#     angle = np.mod(np.arctan2(
#         pre_rot[:,1].reshape(256,256),
#         pre_rot[:,0].reshape(256,256)),np.pi/3).reshape(-1)
    
    
#     # combine the data and label for test
#     whole_data_with_rotation = []
#     for i in tqdm(range(noisy_data.shape[0]),leave=True, total=noisy_data.shape[0]):
#         whole_data_with_rotation.append([noisy_data[i], angle[i]])
        
    return noisy_data

# Loading Data

In [15]:
# Set data direction
data_dir = os.path.abspath("Extremely_Noisy_4DSTEM_Strain_Mapping_Using_CC_ST_AE_Simulated/polycrystal_output4D.mat")
folder_name = ''

In [16]:
data_dir

'/home/shuyu/4DSTEM/Simulated_4dstem/Extremely_Noisy_4DSTEM_Strain_Mapping_Using_CC_ST_AE_Simulated/polycrystal_output4D.mat'

# Integrate Label 

In [17]:
folder_name = folder_name = 'Extremely_Noisy_4DSTEM_Strain_Mapping_Using_CC_ST_AE_Simulated'
label_rotation_path = folder_name +'/Label_rotation.npy'
label_xx_path = folder_name +'/Label_strain_xx.npy'
label_yy_path = folder_name +'/Label_strain_yy.npy'
label_xy_path = folder_name +'/Label_shear_xy.npy'

In [18]:
label_xx_path

'Extremely_Noisy_4DSTEM_Strain_Mapping_Using_CC_ST_AE_Simulated/Label_strain_xx.npy'

In [19]:
label_xx = np.load(label_xx_path).reshape(-1)
label_yy = np.load(label_yy_path).reshape(-1)
label_xy = np.load(label_xy_path).reshape(-1)
label_rot = np.load(label_rotation_path).reshape(-1)

In [20]:
label_cat = np.stack([label_xx,label_yy,label_xy,label_rot])

In [21]:
label_cat = np.transpose(label_cat,(1,0))

In [22]:
label_cat.shape

(65536, 4)

In [23]:
label_cat[0]

array([-0.04201891, -0.02528346, -0.02672141,  1.79829708])

In [24]:
x_train = load_data(data_dir,w_bg=0.25)

100%|██████████████████████████████████████████████████████████████| 65536/65536 [01:36<00:00, 676.78it/s]


In [25]:
x_train.shape

(65536, 1, 200, 200)

In [26]:
def train_with_label(x,y):
    x_with_y = []
    for i in tqdm(range(x.shape[0]),leave=True, total=x.shape[0]):
        x_with_y.append([x[i], y[i]])
    return x_with_y

In [27]:
train_set = train_with_label(x_train,label_cat)

100%|███████████████████████████████████████████████████████████| 65536/65536 [00:00<00:00, 178867.30it/s]


In [28]:
train_set[0][1].shape

(4,)

In [29]:
del(x_train)

In [30]:
device = torch.device('cuda')

# ResNet 50

In [31]:
resnet50 = models.resnet50(pretrained=True)
num_ftrs = resnet50.fc.in_features

In [32]:
num_ftrs

2048

In [33]:
resnet50.fc = nn.Linear(num_ftrs, 4)

In [34]:
resnet50.conv1 = nn.Conv2d(
            1, 64, kernel_size=(7,7), stride=(2,2), padding=(3,3), bias=False
        )

In [35]:
resnet50

ResNet(
  (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

## Loss Function

In [36]:
# def loss_function_strain(model,
#                       train_iterator,
#                       optimizer,
#                       device,
#                      ):

#     # set the train mode
#     model.train()

#     # loss of the epoch
#     train_loss = 0
    
#     for x,y in tqdm(train_iterator, leave=True, total=len(train_iterator)):
     

#         x = x.to(device, dtype=torch.float)
#         y = y.to(device, dtype=torch.float)
#         optimizer.zero_grad()
        

#         y_pred = model(x)
        
#         loss = F.mse_loss(y,y_pred)
    
#         train_loss += loss.item()
#         loss.backward()
#         # update the weights
#         optimizer.step()

#     return train_loss

# Train Function

In [37]:
folder_path = 'Resnet50_25Per_4dstem' 

In [38]:
# def train(data_,
#          model,
#          optimizer,
#          epochs=5000,   
#          learning_rate = 1e-4,
#          max_rate = 1e-3,
#          batch_size = 64,
#          epoch_ = None,
#          file_path = None,
#          folder_path=folder_path,
#          step_size_up=50,
#          best_train_loss= None,
#          set_scheduler = True,
#         ):
        
#             make_folder(folder_path)            
#             device = "cpu"
#             if torch.cuda.is_available():
#                 device = "cuda"
#             seed = 42
#             random.seed(seed)
#             np.random.seed(seed)
#             torch.manual_seed(seed)
#             if torch.cuda.is_available():
#                 torch.cuda.manual_seed(seed)
#             # put model on device
#             model.to(device)
#             print('.........b step.........')
#             patience = 0


#             if set_scheduler:

#                 lr_scheduler = torch.optim.lr_scheduler.CyclicLR(optimizer, base_lr=learning_rate, max_lr=max_rate,
#                                                       step_size_up=step_size_up,cycle_momentum=False)
#             else: 

#                 lr_scheduler = None


#             print('..........successfully generate model')

#             train_iterator = DataLoader(data_, batch_size=batch_size, shuffle=True, num_workers=0)


#             N_EPOCHS = epochs

#             if best_train_loss == None:
#                 best_train_loss = float('inf')


#             if epoch_==None:
#                 start_epoch = 0
#             else:
#                 start_epoch = epoch_+1
#             print('...........successfully generate train interator')

#             for epoch in range(start_epoch,epochs):

#                 optimizer.param_groups[0]['lr'] = learning_rate   

#                 train = loss_function_strain(model,train_iterator,
#                                       optimizer,device)

#                 input_length = len(train_iterator)


#                 train_loss = train

#                 train_loss /= input_length

#         #        VAE_L /= len(train_iterator)
#                 print(f'Epoch {epoch}, Train Loss: {train_loss:.4f}')
#         #        print(f'......... VAE Loss: {VAE_L:.4f}')
#                 print('.............................')

#                 checkpoint = {
#                     "net":model.state_dict(),
#                     'optimizer': optimizer.state_dict(),
#                     "epoch": epoch,
#                     'trainloss': train_loss,
#                 }
#                 if epoch >=0:
#                     lr_ = optimizer.param_groups[0]['lr']
#     #                    l1_form = format(coef_1,'.4f')
#                     file_path = folder_path+f'/0215_25Per_epoch:{epoch:04d}_lr:{lr_:.6f}_trainloss:{train_loss:.6f}_.pkl'

#                     if best_train_loss >= train_loss:
#                         best_train_loss = train_loss
#                         torch.save(checkpoint, file_path)
#                     else:
#                         patience+=1
                        
#                         if patience >=100:
#                             break


                                

#                 if lr_scheduler!= None:
#                     lr_scheduler.step()
                    
#                 if epoch==epochs-1:
#                     del model

In [74]:
def eval_matx(pred_y,
              label_xx,
              label_yy,
              label_xy,
              label_rot,
              ref_region = (30,60,10,40),
             ):

    exx_resnet = pred_y[:,0]
    eyy_resnet = pred_y[:,1]
    exy_resnet = pred_y[:,2]
    rot_resnet = pred_y[:,3].reshape(256,256)
    mae_xx = np.mean(abs(exx_resnet.reshape(-1) - label_xx))
    mae_yy = np.mean(abs(eyy_resnet.reshape(-1) - label_yy))
    mae_xy = np.mean(abs(exy_resnet.reshape(-1) - label_xy))
    
    # create label value of rotation
    label_rot = label_rot.reshape(256,256)
    label_ref_rotation = np.mean(label_rot[ref_region[0]:ref_region[1],
                                            ref_region[2]:ref_region[3]])
    # calculate corresponding rotation based on reference 
    label_rot = label_rot - label_ref_rotation
    # create correct format of rotation for autoencoder
    # calculate rotation autoencoder
    
    rot_resnet_ref = np.mean(rot_resnet[ref_region[0]:ref_region[1],
                                                ref_region[2]:ref_region[3]])
    # calculate corresponding rotation based on reference 
    rot_resnet = rot_resnet - rot_resnet_ref
    
    combine_loss = mae_xx + mae_yy + mae_xy
    combine_loss_with_rot = combine_loss+ np.mean(abs(rot_resnet-label_rot))
    return combine_loss,combine_loss_with_rot

In [51]:
def evaluate(data_, 
             model,
             device = device,
             batch_size = 64,
             label_xx = label_xx,
             label_yy = label_yy,
             label_xy = label_xy,
             label_rot = label_rot):
    model.to(device)
    model.eval()
    test_iterator = DataLoader(data_, batch_size=batch_size, shuffle=False, num_workers=0)
    with torch.no_grad():
            # loss of the epoch
#        train_loss = 0
        pred_y = []
        for x,y in tqdm(test_iterator, leave=True, total=len(test_iterator)):
         
    
            x = x.to(device, dtype=torch.float)
#            y = y.to(device, dtype=torch.float)
            
            y_pred = model(x)
            pred_y.append(y_pred.cpu().detach().numpy())
            # loss = F.mse_loss(y,y_pred)
            # train_loss += loss.item()
        pred_y = np.concatenate(pred_y,axis=0)
#        train_loss /= len(test_iterator)
        combine_loss,combine_loss_with_rot = eval_matx(pred_y,
                                              label_xx,
                                              label_yy,
                                              label_xy,
                                              label_rot)
        return combine_loss,combine_loss_with_rot
        

In [52]:
check_point = torch.load(f'{folder_path}/0215_25Per_epoch:0000_lr:0.000100_trainloss:0.009804_.pkl')
resnet50.load_state_dict(check_point['net'])

<All keys matched successfully>

In [53]:
combine_loss,combine_loss_with_rot = evaluate(train_set,resnet50)

100%|███████████████████████████████████████| 1024/1024 [07:40<00:00,  2.22it/s]
